In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/mac/Studying/OTTO/scenario_generator


In [ ]:
# CAN Encoder demonstration
from src.config_loader import ConfigLoader
from src.signal_generator import Scenario, SignalGenerator
from src.can_encoder import CANEncoder

loader = ConfigLoader("../config")
gen = SignalGenerator(loader)
vehicles = loader.load_vehicles()
signals_cfg = loader.load_signals()

# --- Generate signals for City Drive (sedan_a) ---
scenario = Scenario.from_yaml("../config/scenarios/city_drive.yaml")
df = gen.generate(scenario)

# --- Encode with sedan_a profile ---
encoder = CANEncoder(vehicles["sedan_a"], signals_cfg)
frames_list = encoder.encode_dataframe(df)

# --- Inspect the first 5 frames ---
print(f"Generated {len(frames_list)} time steps")
print(f"CAN IDs emitted per step: {sorted(frames_list[0].keys())}")
print()

for i in [0, 100, 500, 1500, 3000]:
    print(f"--- t = {df['time_s'].iloc[i]:.2f} s ---")
    for can_id, payload in frames_list[i].items():
        print(f"  0x{can_id:03X}: {payload.hex()}")
    print()

# --- Round-trip verification: decode the first frame ---
print("--- Round-trip check (first frame) ---")
decoded = {}
for can_id, payload in frames_list[0].items():
    decoded.update(encoder.decode(can_id, payload))

for sig in encoder.mapped_signal_names():
    original = df.iloc[0][sig]
    recovered = decoded[sig]
    print(f"  {sig:30s}  original={original:10.3f}  decoded={recovered:10.3f}")

# --- Cross-vehicle check: same signal values, different CAN IDs ---
print()
print("--- Same signals encoded for SUV (different CAN IDs) ---")
encoder_suv = CANEncoder(vehicles["suv_b"], signals_cfg)
frames_suv = encoder_suv.encode_dataframe(df)

print(f"  sedan_a CAN IDs: {sorted(frames_list[0].keys())}")
print(f"  suv_b   CAN IDs: {sorted(frames_suv[0].keys())}")
print(f"  Payload lengths: sedan={[len(p) for p in frames_list[0].values()]}, "
      f"suv={[len(p) for p in frames_suv[0].values()]}")

Generated 6000 time steps
CAN IDs emitted per step: [256, 288, 416]

--- t = 0.00 s ---
  0x100: 0c80003c00000000
  0x1A0: 0000000000000000
  0x120: 000003e800000000

--- t = 1.00 s ---
  0x100: 0c80003c00000000
  0x1A0: 0000000000000000
  0x120: 000003e800000000

--- t = 5.00 s ---
  0x100: 20bb2d3d00000000
  0x1A0: 00000007c3000000
  0x120: 07c304fa00000000

--- t = 15.00 s ---
  0x100: 48ff003f00000000
  0x1A0: 0000231d2f000000
  0x120: 1d2f031300000000

--- t = 30.00 s ---
  0x100: 0d87284300000000
  0x1A0: 0000000009000000
  0x120: 000904db00000000

--- Round-trip check (first frame) ---
  engine_rpm                      original=   800.000  decoded=   800.000
  throttle_position               original=     0.000  decoded=     0.000
  coolant_temperature             original=    20.000  decoded=    20.000
  brake_pressure                  original=     0.000  decoded=     0.000
  brake_pedal                     original=     0.000  decoded=     0.000
  wheel_speed                 